In [37]:
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
import torchvision
from torch.nn import init

from torchvision.models.resnet import BasicBlock, ResNet
from torchvision.transforms import ToTensor

In [ ]:
import io
from torchvision import models, transforms
import torch.utils.data as data_utils
from PIL import Image
import os
import scipy.io as sio

import cv2
import matplotlib.pyplot as plt
import torch.nn.functional as F
def default_loader(path):
    return Image.open(path).convert('L')   

In [ ]:
from torchvision.models.resnet import BasicBlock, ResNet
from torch.nn import init

def conv(in_planes, out_planes, kernel_size=3, stride=1, dilation=1, bias=False, transposed=False):
    if transposed:
        layer = nn.ConvTranspose2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=1, output_padding=1,
                                   dilation=dilation, bias=bias)
    else:
        padding = (kernel_size + 2 * (dilation - 1)) // 2
        layer = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias)
    if bias:
        init.constant_(layer.bias, 0)
    return layer

# Returns 2D batch normalisation layer
def bn(planes):
    layer = nn.BatchNorm2d(planes)
    # Use mean 0, standard deviation 1 init
    init.constant_(layer.weight, 1)
    init.constant_(layer.bias, 0)
    return layer


class FeatureResNet(ResNet):
    def __init__(self):
        super().__init__(BasicBlock, [3, 14, 16, 3], 1000)
        self.conv_f = conv(2,64, kernel_size=3,stride = 1)
        self.ReLu_1 = nn.ReLU(inplace=True)
        self.conv_pre = conv(512, 1024, stride=2, transposed=False)
        self.bn_pre = bn(1024)

    def forward(self, x):
        x1 = self.conv_f(x)
        #print('x1',x1.size())
        x = self.bn1(x1)
        #print(x.size())
        x = self.relu(x)
        #print(x.size())
        x2 = self.maxpool(x)
        #print('x2',x2.size())
        x = self.layer1(x2)
        #print(x2.size())
        x3 = self.layer2(x)
        #print('x3',x3.size())
        x4 = self.layer3(x3)
        #print('x4',x4.size())
        x5 = self.layer4(x4)
        #print('x5',x5.size())
        x6 = self.ReLu_1(self.bn_pre(self.conv_pre(x5)))
        #print('x6',x6.size())
        return x1, x2, x3, x4, x5,x6


class SegResNet(nn.Module):
    def __init__(self, num_classes, pretrained_net):
        super().__init__()
        self.pretrained_net = pretrained_net
        self.relu = nn.ReLU(inplace=True)
        #self.conv3 = conv(1024,1024, stride=1, transposed=False)
        #self.bn3 = bn(1024)
        self.conv3_2 = conv(1024, 512, stride=1, transposed=False)
        self.bn3_2 = bn(512)
        self.conv4 = conv(512,512, stride=2, transposed=True)
        self.bn4 = bn(512)
        self.conv5 = conv(512, 256, stride=2, transposed=True)
        self.bn5 = bn(256)
        self.conv6 = conv(256, 128, stride=2, transposed=True)
        self.bn6 = bn(128)
        self.conv7 = conv(128, 64, stride=2, transposed=True)
        self.bn7 = bn(64)
        self.conv8 = conv(64, 64, stride=2, transposed=True)
        self.bn8 = bn(64)
        self.conv9 = conv(64, 32, stride=1, transposed=False)
        self.bn9 = bn(32)
        self.convadd = conv(32, 16, stride=1, transposed=False)
        self.bnadd = bn(16)
        self.conv10 = conv(16, num_classes,stride=2, kernel_size=3)
        init.constant_(self.conv10.weight, 0)  # Zero init

    def forward(self, x):
        #b,c,w,h = x.size()
        #x = x.view(b,c,w,h)
        x1, x2, x3, x4, x5, x6 = self.pretrained_net(x)
        #x1 = x1.view(b,x1.size(1),x1.size(2),x1.size(3))
        #x2 = x2.view(b,x2.size(1),x2.size(2),x2.size(3))
        #x3 = x3.view(b,x3.size(1),x3.size(2),x3.size(3))
        #x4 = x4.view(b,x4.size(1),x4.size(2),x4.size(3))
        #x5 = x5.view(b,x5.size(1),x5.size(2),x5.size(3))

        #x1 = torch.max(x1,1)[0]
        #x2 = torch.max(x2,1)[0]
        #x3 = torch.max(x3,1)[0]
        #x4 = torch.max(x4,1)[0]
        #x5 = torch.max(x5,1)[0]


        #print(x5.size())
        #print(x4.size())
        #print(x1.size())
        #x = self.relu(self.bn3(self.conv3(x6)))
        x = self.relu(self.bn3_2(self.conv3_2(x6)))
        
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))
        #print(x.size())
        x = self.relu(self.bn6(self.conv6(x+x4 )))
        #print(x.size())
        x = self.relu(self.bn7(self.conv7(x+x3 )))
        #print(x.size())
        x = self.relu(self.bn8(self.conv8(x+x2 )))
        #print(x.size())
        x = self.relu(self.bn9(self.conv9(x+x1 )))
        #print(x.size())
        x = self.relu(self.bnadd(self.convadd(x)))
        x = self.conv10(x)
        return x


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

fnet = FeatureResNet()
fcn = SegResNet(4,fnet)
fcn = fcn.to(device)


In [ ]:
dataset_path = '.'

# The image numbers for each split come from these two files. They do not overlap,
# so validation images are never trained on.
def read_ids(filename):
    with open(filename) as f:
        return [line.strip() for line in f if line.strip()]

def make_triplets(ids):
    triplets = []
    for i in ids:
        triplet = (
            os.path.join(dataset_path, 'imgs3', 'train_image_' + i + '_1.png'),
            os.path.join(dataset_path, 'imgs3', 'train_image_' + i + '_2.png'),
            os.path.join(dataset_path, 'gt3', 'train_image_' + i + '.mat')
        )
        # Skip samples that are not on disk, so a partial download still trains
        if all(os.path.exists(path) for path in triplet):
            triplets.append(triplet)
    return triplets

test_set = make_triplets(read_ids('validation_dataset.txt'))
train_set = make_triplets(read_ids('train_dataset.txt'))

print('train set length:', len(train_set))
print('test set length:', len(test_set))

if not train_set or not test_set:
    raise FileNotFoundError('No samples found. Put the dataset from the README link in imgs3 and gt3 under ' + os.path.abspath(dataset_path))


In [ ]:
import scipy.io as sio
from scipy import interpolate
x = np.arange(0,256,1)
y = np.arange(0,256,1)
xnew = np.arange(1.5,257.5,4)
ynew = np.arange(1.5,257.5,4)

class MyDataset(data_utils.Dataset):
    def __init__(self, dataset, transform=None, target_transform=None, loader=default_loader):
       
        self.imgs = dataset
        self.transform = transform
        self.target_transform = target_transform
        self.loader = loader

    def __getitem__(self, index):
        label_x, label_y, label_z = self.imgs[index]
        img1 = self.loader(label_x)
        img_1 = ToTensor()(img1.resize((128,128)))
        img2 = self.loader(label_y)
        img_2 = ToTensor()(img2.resize((128,128)))
        imgs = torch.cat((img_1, img_2), 0)
        try:
            gt = sio.loadmat(label_z)['Disp_field_1'].astype(float)
            
        except KeyError:
            gt = sio.loadmat(label_z)['Disp_field_2'].astype(float)
           
        gt = np.asarray(gt)
        gt = gt*100
        # np.gradient returns d/d(row) first, then d/d(column). This channel order is the one the
        # published models were trained on. With channel 0 horizontal (see Diaplacement_run):
        # dudx = du/d(row), dudy = du/d(column), dvdx = dv/d(row), dvdy = dv/d(column)
        [dudx, dudy]= np.gradient(gt[:,:,0])
        [dvdx, dvdy]= np.gradient(gt[:,:,1])
        
        dudx_ = interpolate.RectBivariateSpline(y, x, dudx)(ynew, xnew)
        dudy_ = interpolate.RectBivariateSpline(y, x, dudy)(ynew, xnew)
        dvdx_ = interpolate.RectBivariateSpline(y, x, dvdx)(ynew, xnew)
        dvdy_ = interpolate.RectBivariateSpline(y, x, dvdy)(ynew, xnew)
        st = np.stack([dudx_, dudy_, dvdx_, dvdy_], axis=0)
                #st = np.stack([dudx, dudy, dvdx, dvdy], axis=0)
        
        
        return imgs,torch.from_numpy(st).float()

    def __len__(self):
        return len(self.imgs)

In [43]:
EPOCH = 200              # train the training data n times, 200 epochs as in the paper
BATCH_SIZE = 12
print('BATCH_SIZE = ',BATCH_SIZE)
LR = 0.001              # learning rate
#root = './gdrive_northwestern/My Drive/dl_encoder/data/orig/orig'
NUM_WORKERS = 0

optimizer = torch.optim.Adam(fcn.parameters(), lr=LR)   # optimize all cnn parameters
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[100], gamma=0.01)   # lr 1e-3 -> 1e-5 after 100 epochs, as in the paper
#optimizer = torch.optim.SGD(cnn.parameters(), lr=LR, momentum=0.9)   # optimize all cnn parameters
loss_func = nn.MSELoss()


train_data=MyDataset(dataset=train_set)
train_loader = data_utils.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

test_data=MyDataset(dataset=test_set)
test_loader = data_utils.DataLoader(dataset=test_data, batch_size=1)

BATCH_SIZE =  12


In [ ]:
from datetime import datetime
dataString = datetime.strftime(datetime.now(), '%Y_%m_%d_%H_%M_%S')

In [ ]:
from pathlib import Path

# Give your root folder a name
root_result = Path("results") 
log_result = root_result / "log"
model_result = root_result / "model"

# .mkdir(parents=True) creates the root and subfolders all at once
# exist_ok=True prevents an error if the folders already exist
model_result.mkdir(parents=True, exist_ok=True)
log_result.mkdir(parents=True, exist_ok=True)

print(f"Directories created at: {root_result.absolute()}")

In [ ]:
# Initialize Log Files
fileOut = open(os.path.join(log_result, 'log' + dataString), 'a')
fileOut.write(dataString + ' Epoch:   Step:    Loss:        Val_Accu :\n')
fileOut.close()

fileOut2 = open(os.path.join(log_result, 'validation' + dataString), 'a')
fileOut2.write('kernel_size of conv_f is 3\n')
fileOut2.write(dataString + ' Epoch:    loss:\n')
fileOut2.close()

# 2. TRAINING LOOP
for epoch in range(EPOCH):
    fcn.train()
    for step, (img, gt) in enumerate(train_loader):
        # Move data to the device (.to(device) replaces .cuda())
        img = img.to(device)
        gt = gt.float().to(device)
        
        output = fcn(img)
        loss = loss_func(output, gt)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(epoch, step, loss.item())
        
        # Write training logs
        fileOut = open(os.path.join(log_result, 'log' + dataString), 'a')
        fileOut.write(str(epoch) + '   ' + str(step) + '   ' + str(loss.item()) + '\n')
        fileOut.close()

    scheduler.step()

    # Save checkpoints every 10 epochs
    if epoch % 10 == 9:
        PATH = os.path.join(model_result, f'param_all_strain2_{epoch}_{step}.pth')
        torch.save(fcn.state_dict(), PATH)
        print('Finished saving checkpoints')

    # 3. VALIDATION LOOP
    LOSS_VALIDATION = 0
    fcn.eval()
    with torch.no_grad():
        # Changed 'step' to 'v_step' here to avoid confusion with training steps
        for v_step, (img, gt) in enumerate(test_loader):
            img = img.to(device)
            gt = gt.to(device)
            
            output = fcn(img)
            LOSS_VALIDATION += loss_func(output, gt)
            
        # Calculate average loss
        LOSS_VALIDATION = LOSS_VALIDATION / (v_step + 1)
        
        # Write validation logs
        fileOut2 = open(os.path.join(log_result, 'validation' + dataString), 'a')
        fileOut2.write(str(epoch) + '   ' + str(v_step) + '   ' + str(LOSS_VALIDATION.item()) + '\n')
        fileOut2.close()
        
        print(f'Validation error epoch {epoch}: {LOSS_VALIDATION.item()}')

In [23]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")

Is CUDA available? False
